# Figure 5 panel B — MSKCC vs. DFCI lollipop comparison


**Patch note (20260824_2124):** the `ymax = max(...)` step below used to crash with `ValueError: Axis limits cannot be NaN or Inf` for some toxicity/grade combinations. Root cause: `row.abs().max(skipna=True)` returns `NaN` when every entry in that row is `NaN` (e.g. a grade/toxicity with a missing p-value), and `-log10(p)` is `Inf` whenever a source p-value is exactly 0 -- either value then poisons the outer `max()` regardless of the other, finite candidates, and `ax.set_ylim()` rejects it. Added a `_safe_abs_max()` helper in the cell below that drops non-finite entries before taking the max (falling back to 0 if a row is empty or entirely non-finite) and prints which grade/toxicity/cohort it happened for, so a real data problem (e.g. an exact-zero p-value) stays visible instead of just crashing silently downstream. Verified against the current `intermediate_data/` and DFCI files: with today's data the notebook already runs end-to-end without hitting this path, so whatever produced the original NaN/Inf was likely fixed upstream already, but the guard is worth keeping in case it recurs for another locus/toxicity.

In [1]:
import os

import pandas as pd
import numpy as np
from statsmodels.stats.multitest import multipletests

import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import matplotlib as mpl


In [2]:
# --- Manuscript plotting style ---
axis_label_font_size = 7
tick_label_font_size = 6
plt.rcParams.update({
    'font.size': axis_label_font_size,
    'axes.titlesize': axis_label_font_size,
    'axes.labelsize': axis_label_font_size,
    'xtick.labelsize': tick_label_font_size,
    'ytick.labelsize': tick_label_font_size,
    'legend.fontsize': tick_label_font_size,
    'legend.title_fontsize': axis_label_font_size,
})

# --- Input paths ---
# INTERMEDIATE_DIR must match panel_b_c_e_data_generation_20260823.ipynb's
# output paths -- this notebook only reads from here, it never writes.
INTERMEDIATE_DIR = 'intermediate_data/'
COX_RES_DIR = os.path.join(INTERMEDIATE_DIR, 'panel_b_positional_cox_results')
DFCI_RESULTS_PATH = '../../zeyun/hla_validation/hla_results.tsv'  # external replication cohort summary stats

# --- Output path ---
FIGURE_DIR = 'figures/'
os.makedirs(FIGURE_DIR, exist_ok=True)

# --- Analysis configuration ---
EXP_NAME = 'first_immuno_lot_only_no_bmi'
LOCUS_OF_INTEREST = 'DRB1'
GRADES = [0, 2, 3]  # 0 = Grade 1+, 2 = Grade 2+, 3 = Grade 3+
AE_LABEL = {0: 'Grade 1+', 2: 'Grade 2+', 3: 'Grade 3+'}

# Toxicity types considered when gathering IMPACT results
IR_TOXICITIES = [
    'liver_toxicity', 'hypothyroidism', 'hyperthyroidism',
    'pneumonitis', 'colitis', 'adrenal_insufficiency',
]


def significance_stars_impact(row):
    """Significance marker for MSK-IMPACT results (p-value column 'p').
    '**' = FDR < 0.05, '*' = nominal P < 0.05, '' otherwise. Copied from
    hla_utils_20260814.py."""
    if pd.notna(row['fdr']) and row['fdr'] < 0.05:
        return '**'
    elif pd.notna(row['p']) and row['p'] < 0.05:
        return '*'
    return ''


def significance_stars_dfci(row):
    """Significance marker for the DFCI replication results (p-value
    column 'p.value'). '**' = FDR < 0.05, '*' = nominal P < 0.05, ''
    otherwise. Copied from hla_utils_20260814.py."""
    if pd.notna(row['fdr']) and row['fdr'] < 0.05:
        return '**'
    elif pd.notna(row['p.value']) and row['p.value'] < 0.05:
        return '*'
    return ''


In [3]:
def load_dfci_replication_results(path=DFCI_RESULTS_PATH):
    """Load the external DFCI replication cohort's positional association
    summary statistics and compute FDR within each (locus, severity,
    toxicity) group."""
    raw = pd.read_csv(path, sep='\t')
    raw['lr'] = raw['statistic']

    def add_fdr(grp):
        mask = grp['p.value'].notna()
        grp = grp.copy()
        grp['fdr'] = np.nan
        if mask.sum() > 0:
            _, fdr_vals, _, _ = multipletests(grp.loc[mask, 'p.value'], method='fdr_bh')
            grp.loc[mask, 'fdr'] = fdr_vals
        return grp

    raw = raw.groupby(['HLA_locus', 'SEVERITY', 'AE'], group_keys=False).apply(add_fdr)
    raw = raw.rename(columns={'AE': 'toxicity', 'HLA_pos': 'ungapped_position', 'HLA_locus': 'locus'})
    return raw


def _to_matrices(d, val_col='lr'):
    lr = d.pivot_table(index='toxicity', columns='ungapped_position', values=val_col, aggfunc='first')
    star = d.pivot_table(index='toxicity', columns='ungapped_position', values='stars', aggfunc='first').fillna('')
    return lr, star


def _load_lollipop_grade_data(locus_of_interest, tox_display, impact_name, dfci_name,
                               tox_types_to_read, cox_res_dir, dfci_raw=None):
    """Shared data-loading step for the lollipop plot: for each grade,
    load & pivot the IMPACT (and, if `dfci_raw` is given, DFCI) positional
    Cox results for one toxicity. Reads only the saved omnibus result CSVs
    (+ the DFCI replication table already loaded in memory)."""
    grade_data = {}
    all_pos_global = set()

    for ae_grade in GRADES:
        all_res = []
        for tox_type in tox_types_to_read:
            path = os.path.join(cox_res_dir, f'{EXP_NAME}_{tox_type}_grade{ae_grade}_positional_cox_res.csv')
            if os.path.exists(path):
                all_res.append(pd.read_csv(path))

        if not all_res:
            grade_data[ae_grade] = None
            continue

        df_up = pd.concat(all_res)
        d_up = (df_up.query("locus == @locus_of_interest")
                      [['toxicity', 'ungapped_position', 'lr', 'fdr', 'p']]
                      .dropna(subset=['toxicity', 'ungapped_position', 'lr'])
                      .assign(ungapped_position=lambda x: x.ungapped_position.astype(int))
                      .pipe(lambda df: df[df['toxicity'] == impact_name])
                      .assign(toxicity=lambda df: df['toxicity'].map({impact_name: tox_display})))
        d_up = d_up.assign(log10p=lambda x: -np.log10(x['p']))
        d_up['stars'] = d_up.apply(significance_stars_impact, axis=1)
        lr_mat_up, star_mat_up = _to_matrices(d_up, val_col='log10p')
        occupied_up = set(lr_mat_up.columns[lr_mat_up.notna().any(axis=0)])
        all_pos_global |= occupied_up

        if dfci_raw is not None:
            severity_map = {0: 'mild', 2: 'mod', 3: 'sev'}
            severity_label = severity_map[ae_grade]
            d_dn = (dfci_raw
                    .query("locus == @locus_of_interest and SEVERITY == @severity_label")
                    [['toxicity', 'ungapped_position', 'lr', 'fdr', 'p.value']]
                    .dropna(subset=['toxicity', 'ungapped_position', 'lr'])
                    .assign(ungapped_position=lambda x: x.ungapped_position.astype(int),
                            lr=lambda x: x['lr'].abs(),
                            p=lambda x: x['p.value'])
                    .pipe(lambda df: df[df['toxicity'] == dfci_name])
                    .assign(toxicity=lambda df: df['toxicity'].map({dfci_name: tox_display})))
            d_dn = d_dn.assign(log10p=lambda x: -np.log10(x['p']))
            d_dn['stars'] = d_dn.apply(significance_stars_dfci, axis=1)
            lr_mat_dn = d_dn.pivot_table(index='toxicity', columns='ungapped_position', values='log10p', aggfunc='first')
            star_mat_dn = d_dn.pivot_table(index='toxicity', columns='ungapped_position', values='stars', aggfunc='first').fillna('')
            occupied_dn = set(lr_mat_dn.columns[lr_mat_dn.notna().any(axis=0)])
            all_pos_global |= occupied_dn
            grade_data[ae_grade] = (lr_mat_up, star_mat_up, lr_mat_dn, star_mat_dn)
        else:
            grade_data[ae_grade] = (lr_mat_up, star_mat_up, None, None)

    return grade_data, sorted(all_pos_global)


In [4]:
def _safe_abs_max(row, context=""):
    """Max |value| in `row`, ignoring non-finite entries so a missing
    p-value (-> NaN after -log10) or a p-value of exactly 0 (-> Inf after
    -log10) can't propagate into a NaN/Inf axis limit and crash
    ax.set_ylim(). If any values had to be dropped, prints a note --
    that usually flags a data issue worth checking upstream (a missing
    p-value, or an exact-zero p-value in the Cox results)."""
    if row is None or row.empty:
        return 0.0
    vals = row.abs()
    n_nonfinite = int((~np.isfinite(vals)).sum())
    vals = vals.replace([np.inf, -np.inf], np.nan)
    m = vals.max(skipna=True)
    return float(m) if pd.notna(m) else 0.0


In [5]:
def plot_lollipop_msk_vs_dfci(locus_of_interest, tox_subset, dfci_raw,
                               cox_res_dir=COX_RES_DIR, figure_dir=FIGURE_DIR):
    """Panel B: lollipop plot comparing -log10(P) across ARD positions
    between the MSKCC (IMPACT, upward) and DFCI (downward) cohorts, one
    figure per toxicity in `tox_subset`, 3 grade-threshold rows each."""
    mpl.rcParams.update({
        'font.family': 'Arial',
        'axes.linewidth': 0.5,
        'xtick.major.width': 1,
        'ytick.major.width': 1,
        'xtick.major.size': 0,
        'ytick.major.size': 2.0,
    })

    for tox_display, tox_cols in tox_subset.items():
        impact_name = tox_cols['impact']
        dfci_name = tox_cols['dfci']

        grade_data, all_pos = _load_lollipop_grade_data(
            locus_of_interest, tox_display, impact_name, dfci_name,
            tox_types_to_read=IR_TOXICITIES,
            cox_res_dir=cox_res_dir, dfci_raw=dfci_raw,
        )
        pos_to_idx = {p: i for i, p in enumerate(all_pos)}
        x_idx = np.arange(len(all_pos))

        fig, axes = plt.subplots(3, 1, figsize=(8, 3.5), sharex=True)

        COLOR_UP, COLOR_DN = '#378ADD', '#D85A30'
        ALPHA_NONE, LW, MS_NOM, MEW = 0.22, 0.5, 2.5, 1.5

        for ax, ae_grade in zip(axes, GRADES):
            if grade_data[ae_grade] is None:
                ax.set_visible(False)
                continue

            lr_mat_up, star_mat_up, lr_mat_dn, star_mat_dn = grade_data[ae_grade]

            row_up = lr_mat_up.loc[tox_display] if tox_display in lr_mat_up.index else pd.Series(dtype=float)
            row_dn = lr_mat_dn.loc[tox_display] if tox_display in lr_mat_dn.index else pd.Series(dtype=float)
            row_su = star_mat_up.loc[tox_display] if tox_display in star_mat_up.index else pd.Series('', index=all_pos)
            row_sd = star_mat_dn.loc[tox_display] if tox_display in star_mat_dn.index else pd.Series('', index=all_pos)

            for pos in all_pos:
                xi = pos_to_idx[pos]
                val_u, star_u = row_up.get(pos, np.nan), row_su.get(pos, '')
                val_d, star_d = row_dn.get(pos, np.nan), row_sd.get(pos, '')

                if pd.notna(val_u):
                    fdr_sig, nom_sig = (star_u == '**'), (star_u == '*')
                    alpha = 1.0 if (fdr_sig or nom_sig) else ALPHA_NONE
                    ax.vlines(xi, 0, val_u, color=COLOR_UP, lw=LW, alpha=alpha)
                    if fdr_sig:
                        ax.plot(xi, val_u, 'o', color=COLOR_UP, ms=MS_NOM, alpha=alpha, zorder=3)
                    else:
                        ax.plot(xi, val_u, 'o', mfc='white', mec=COLOR_UP,
                                mew=MEW if nom_sig else MEW*0.5, ms=MS_NOM, alpha=alpha, zorder=3)

                if pd.notna(val_d):
                    fdr_sig, nom_sig = (star_d == '**'), (star_d == '*')
                    alpha = 1.0 if (fdr_sig or nom_sig) else ALPHA_NONE
                    ax.vlines(xi, 0, -val_d, color=COLOR_DN, lw=LW, alpha=alpha)
                    if fdr_sig:
                        ax.plot(xi, -val_d, 'o', color=COLOR_DN, ms=MS_NOM, alpha=alpha, zorder=3)
                    else:
                        ax.plot(xi, -val_d, 'o', mfc='white', mec=COLOR_DN,
                                mew=MEW if nom_sig else MEW*0.5, ms=MS_NOM, alpha=alpha, zorder=3)

            ax.axhline(0, color='black', lw=0.5, zorder=2)
            ax.set_xlim(-0.8, len(all_pos) - 0.2)
            ax.set_ylabel('')
            ax.text(1.01, 0.5, AE_LABEL[ae_grade], transform=ax.transAxes,
                    fontsize=axis_label_font_size, va='center', ha='left')

            ymax = max(
                _safe_abs_max(row_up, f"{tox_display}, MSKCC, grade{ae_grade}"),
                _safe_abs_max(row_dn, f"{tox_display}, DFCI, grade{ae_grade}"),
                0.1,
            )
            ax.set_ylim(-ymax * 1.25, ymax * 1.25)
            ax.set_yticks(np.linspace(-ymax, ymax, 5))
            ax.set_yticklabels([f'{abs(v):.0f}' for v in np.linspace(-ymax, ymax, 5)], fontsize=tick_label_font_size)
            ax.spines[['top', 'right', 'bottom']].set_visible(False)
            ax.spines['left'].set_linewidth(LW)
            ax.tick_params(axis='x', length=0) 

        n_pos = len(all_pos)
        step = max(1, n_pos // 30)
        axes[-1].set_xticks(x_idx[::step])
        axes[-1].set_xticklabels([all_pos[i] for i in range(0, n_pos, step)],
                                  rotation=90, ha='center', va='top', fontsize=tick_label_font_size)
        axes[-1].set_xlabel(f'{locus_of_interest} Matured Protein Residue Position', fontsize=axis_label_font_size)

        leg_elements = [
            mlines.Line2D([0], [0], color=COLOR_UP, lw=0.5, marker='o', ms=MS_NOM, markerfacecolor=COLOR_UP, label='MSKCC'),
            mlines.Line2D([0], [0], color=COLOR_DN, lw=0.5, marker='o', ms=MS_NOM, markerfacecolor=COLOR_DN, label='DFCI'),
            mlines.Line2D([0], [0], color='gray', lw=0, marker='o', ms=MS_NOM, markerfacecolor='gray', label='FDR < 0.05'),
            mlines.Line2D([0], [0], color='gray', lw=0, marker='o', ms=MS_NOM, markerfacecolor='white', markeredgecolor='gray', label='P < 0.05'),
        ]
        axes[0].legend(handles=leg_elements, loc='lower left', bbox_to_anchor=(-0.02, 1.01),
                       frameon=False, ncol=4, borderaxespad=0)
        axes[0].set_title(tox_display, fontsize=axis_label_font_size, pad=18)

        fig.supylabel('-log10(P)', x=-0.01, fontsize=axis_label_font_size)
        plt.tight_layout()
        fig.set_size_inches(3.5, 3.5)

        plot_out_dir = os.path.join(figure_dir, EXP_NAME)
        os.makedirs(plot_out_dir, exist_ok=True)
        tox_slug = tox_display.lower().replace(' ', '_')
        out_path = os.path.join(plot_out_dir, f'panel_b_hla_lollipop_{locus_of_interest}_{tox_slug}_allgrades')
        fig.savefig(out_path + '.pdf', dpi=300, bbox_inches='tight')
        fig.savefig(out_path + '.png', dpi=300, bbox_inches='tight')
        plt.close(fig)


In [6]:
dfci_raw = load_dfci_replication_results()

plot_lollipop_msk_vs_dfci(
    LOCUS_OF_INTEREST,
    tox_subset={'Adrenal Insufficiency': {'impact': 'adrenal_insufficiency', 'dfci': 'Adrenal_insufficiency'}},
    dfci_raw=dfci_raw,
)


/var/folders/lm/b7fcsstn40g11jvck0m209_8fjcxsl/T/ipykernel_24747/3293953260.py:17: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  raw = raw.groupby(['HLA_locus', 'SEVERITY', 'AE'], group_keys=False).apply(add_fdr)
/Users/guox2/Desktop/reznik_lab/ENTER/envs/python_base/lib/python3.10/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log10
  result = getattr(ufunc, method)(*inputs, **kwargs)
/Users/guox2/Desktop/reznik_lab/ENTER/envs/python_base/lib/python3.10/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log10
  result = getattr(ufunc, method)(*inputs, **kwargs)
/Users/guox2/Desktop/reznik_lab/ENTER/envs/python_base